In [1]:
%load_ext autoreload
%autoreload 2
import os
import sys
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = '1'
import numpy as np
import pandas as pd
from typing import Dict, List
# from dotenv import load_dotenv  # pip install python-dotenv
from dynclim.features.safety import SegmentationConfig, ThresholdConfig, CompositeConfig, diagnose_feature_completeness, summarize_all_factors, domain_failure_detail, segment_starvation_detail
from dynclim.experiments.runner import ExperimentRunner
from dynclim.data import HealthcareDataPipeline
from dynclim.utils import loadDomainsFromYaml
import arrow
# from dynclim.data.database import get_reports_from_db, create_database_engine
# load_dotenv(dotenv_path='/dynClimToolkit/.env')
# load_dotenv()
from pathlib import Path
# If your notebook is in the project root
project_root = Path("/Users/mrosen44/Documents/dev/DynamicNarrativeClimateToolkit")



/Users/mrosen44/Documents/dev/DynamicNarrativeClimateToolkit/.dynClimVenv/lib/python3.9/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
2026-02-17 21:28:40.676 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-02-17 21:28:40.677 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-02-17 21:28:40.678 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


In [2]:
# prepare domains and data for feature extraction
domain_definitions = loadDomainsFromYaml(project_root / 'config/domains/hch_domains.yaml')

pipeline = HealthcareDataPipeline(
    events_path=project_root / "dynclim/data/raw/hch/JHHCMC HERO Event Data Jan 1 2024 to December 31 2025.xlsx",
    output_dir=project_root / "dynclim/data/processed/hero_hch"
)
results = pipeline.run()
report_data = pipeline.get_report_data()  # [(eid, narrative), ...]

2026-02-17 21:28:43,709 - INFO - Loading healthcare events from /Users/mrosen44/Documents/dev/DynamicNarrativeClimateToolkit/dynclim/data/raw/hch/JHHCMC HERO Event Data Jan 1 2024 to December 31 2025.xlsx
2026-02-17 21:28:45,039 - INFO - Loaded 6,401 rows, 39 columns
2026-02-17 21:28:45,039 - INFO - Column 'event_id': using 'ID' (primary 'Event ID' not found)
2026-02-17 21:28:45,039 - INFO - Column 'event_date': using 'Occurrence Date' (primary 'Event Date' not found)
2026-02-17 21:28:45,040 - INFO - Column 'description': using 'Description' (primary 'Event Description' not found)
2026-02-17 21:28:45,040 - INFO - Resolved columns: {'event_id': 'ID', 'facility': 'Primary Organization', 'unit': 'Primary Location', 'event_date': 'Occurrence Date', 'description': 'Description', 'suggestions': 'Suggestions'}
2026-02-17 21:28:45,308 - INFO - Cleaning: 6,401 -> 6,356 events (45 removed)
2026-02-17 21:28:45,383 - INFO - Saved 6,356 events to /Users/mrosen44/Documents/dev/DynamicNarrativeClimat

In [ ]:
# Run experiment with checkpointing

runner = ExperimentRunner(
    domain_definitions = domain_definitions,
    use_acceleration=True,
    device_pref='auto',
    use_parallel=False,
    embedding_model_names=[      
                                 
        'all-MiniLM-L6-v2',        # small, from sentence transformer team
        # 'all-mpnet-base-v2',     # medium, from sentence transformer team
        'all-roberta-large-v1',  # large, from sentence transformer team
        # 'BAAI/bge-large-en-v1.5' # very large, and generally better performance, but has different training method
                                   # when compared to above models which vary just on size
        # 'mixedbread-ai/mxbai-embed-large-v1',   # mid-sized
        'BAAI/bge-m3'                           # successor to BAAI model above
        ]
)

# Test with subset of configurations
config_filter = {
    #'segmentation_methods': ['sentence'],
    #'threshold_methods': ['top_k'],
    #'composite_methods': ['attention_weight', 'weighted_average'],
    'sentiment_methods': ['vader','siebert'] #['roberta', 'distilbert', 'finbert','vader','siebert']
}

results_df = runner.run_experiment(
    report_data, 
    experiment_name = "hero_hch_newNEW_configs_02-17-2026",
    config_filter=config_filter,
    resume_from_checkpoint=True,
    # save_interval=2,
    chunk_size=None
    # max_workers=10
)

   Registry: dropped 22 constant columns → 12 remaining (constants in registry_constants.json)

🔬 Running experiment 'hero_hch_newNEW_configs_02-17-2026'
   Total configurations: 84
   Pending configurations: 84
   Reports: 6356
   Embedding models: 2
   Parallel processing: False
   Caching: Enabled (embeddings=True, segments=True)


Configurations:   0%|          | 0/84 [00:00<?, ?it/s]


🔄 Initializing processor:
   Embedding model: BAAI/bge-m3
   Sentiment method: siebert
🚀 Using Apple Silicon MPS acceleration


2026-02-17 21:28:52,467 - INFO - Load pretrained SentenceTransformer: BAAI/bge-m3


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Initialized Processor:
   Domains: ['COM', 'SC', 'TWK', 'PS', 'INEX']
   Total items: 5
   Device: mps
   Caching: embeddings=True, segments=True
   Injected sentiment cache: 0 entries

🚀 Using batch-optimized processing...
   Pre-segmenting all texts...
[seg] SentenceSegmenter backend=spacy, min_length=5, unit=chars
[seg] built segmenter SentenceSegmenter for key=('5a5a91d711baaba23988779567c0b9f2', 'chars')
   Found 40685 unique segments
   Pre-encoding unique segments...
   ✅ Pre-cached 40685 embeddings

🚀 Starting processing...
   Reports: 6356
   Parallel: False


... (168 outputs left out)

✅ Processing complete!
   Total time: 37.08 seconds
   Rate: 171.4 reports/second

💾 Checkpoint saved: 83 configurations
   Full results directory: checkpoints/hero_hch_newNEW_configs_02-17-2026/_cfg

🚀 Using batch-optimized processing...
   Pre-segmenting all texts...
   Found 40685 unique segments
   Pre-encoding unique segments...
   ✅ Pre-cached 40685 embeddings

🚀 Starting processing...
   Reports: 6356
   Parallel: False


Configurations: 100%|██████████| 84/84 [2:06:35<00:00, 90.43s/it, cache_hit=99.0%, time=10.3s]

✅ Processing complete!
   Total time: 10.16 seconds
   Rate: 625.6 reports/second

💾 Checkpoint saved: 84 configurations
   Full results directory: checkpoints/hero_hch_newNEW_configs_02-17-2026/_cfg

💾 Checkpoint saved: 84 configurations
   Full results directory: checkpoints/hero_hch_newNEW_configs_02-17-2026/_cfg

✅ Experiment complete!
   Total time: 7595.8 seconds
   Average time per config: 90.43 seconds

📊 Final Cache Statistics:

📊 Cache Performance Report
Device: mps

🎯 Hit Rates:
   Segment Embeddings: 98.8%
   Sentiments: 100.0%
   Text Segmentation: 98.8%
   Similarities: 0.0%
   Overall: 99.0%

📈 Reuse Statistics:
   Unique segments: 40,643
   Total processed: 3,487,302
   Reuse factor: 85.80x

💾 Cache Sizes:
   segment_embeddings: 40,643 entries
   sentiments: 40,643 entries
   text_segments: 6,226 entries
   similarities: 0 entries
📄 Results exported to: checkpoints/hero_hch_newNEW_configs_02-17-2026_results.csv


📄 Results exported to: checkpoints/hero_hch_newNEW_configs_02-17-2026_results.h5

📁 Full report-level results saved in: checkpoints/hero_hch_newNEW_configs_02-17-2026/_cfg
   Each configuration's full results available as separate files


In [4]:
diag_df = diagnose_feature_completeness(
    '/Users/mrosen44/Documents/dev/DynamicNarrativeClimateToolkit/dynclim/notebooks/checkpoints/hero_hch_newNEW_configs_02-17-2026/_cfg'
    # '/Volumes/calculon/event_reporting/rail_full_run_new_configs_02-04-2026/_cfg'
)

Loaded registry: 84 configurations

FEATURE COMPLETENESS DIAGNOSTIC SUMMARY
  Total configurations:    84
  Missing results files:   0
  Configs with any NA:     0 (0.0%)
  Configs with zero var:   0 (0.0%)
  Configs with sparsity:   3 (3.6%)
  Total failure-flagged:   3 (3.6%)
  ---
  Configs with seg data:   84
  Segment starvation:      28 (33.3%)
  Mean segs (starved):     1.8
  Mean segs (healthy):     5.4
  Pct reports ≤1 seg (starved): 41.4%



In [5]:
seg_detail = segment_starvation_detail(
    diag_df,
    factor=['sent_method', 'th__method', 'comp__method']
)

In [6]:
seg_detail

,sent_method,th__method,comp__method,seg__global_mean_mean,seg__global_mean_min,seg__global_median_mean,seg__global_median_min,seg__global_min_mean,seg__global_min_min,seg__global_max_mean,seg__global_max_min,seg__pct_reports_le1_mean,seg__pct_reports_eq0_mean,starvation_rate,failure_rate,n_configs
0,siebert,adaptive,attention_weight,1.84,1.82,1.4,1.4,0.0,0.0,23.3,22.6,0.41355,0.10275,1.0,0.0,6
1,siebert,adaptive,power_attention,1.84,1.82,1.4,1.4,0.0,0.0,23.3,22.6,0.41355,0.10275,1.0,0.0,6
2,siebert,adaptive,weighted_average,1.84,1.82,1.4,1.4,0.0,0.0,23.3,22.6,0.41355,0.10275,1.0,0.0,2
3,vader,adaptive,attention_weight,1.84,1.82,1.4,1.4,0.0,0.0,23.3,22.6,0.41355,0.10275,1.0,0.0,6
4,vader,adaptive,power_attention,1.84,1.82,1.4,1.4,0.0,0.0,23.3,22.6,0.41355,0.10275,1.0,0.5,6
5,vader,adaptive,weighted_average,1.84,1.82,1.4,1.4,0.0,0.0,23.3,22.6,0.41355,0.10275,1.0,0.0,2
6,siebert,top_k,attention_weight,5.39,4.12,5.0,5.0,1.0,1.0,43.0,5.0,0.05790,0.00000,0.0,0.0,12
7,siebert,top_k,power_attention,5.39,4.12,5.0,5.0,1.0,1.0,43.0,5.0,0.05790,0.00000,0.0,0.0,12
8,siebert,top_k,weighted_average,5.39,4.12,5.0,5.0,1.0,1.0,43.0,5.0,0.05790,0.00000,0.0,0.0,4
9,vader,top_k,attention_weight,5.39,4.12,5.0,5.0,1.0,1.0,43.0,5.0,0.05790,0.00000,0.0,0.0,12


In [55]:
pd.DataFrame(diag_df).to_csv("hch_feature_diag.csv")

In [7]:
diag_result = summarize_all_factors(diag_df)
# domain_failure_detail(diag_df)

In [9]:
diag_result.keys()

dict_keys(['embedding_model', 'sent_method', 'seg__min_length', 'comp__method', 'comp__power', 'comp__temperature', 'seg__global_mean', 'seg__pct_reports_le1', 'COM_1_n_segments__mean', 'COM_1_n_segments__pct_le1', 'SC_1_n_segments__mean', 'SC_1_n_segments__pct_le1', 'TWK_1_n_segments__mean', 'TWK_1_n_segments__pct_le1', 'PS_1_n_segments__mean', 'PS_1_n_segments__pct_le1', 'INEX_1_n_segments__mean', 'INEX_1_n_segments__pct_le1', 'failure_flag_with_starvation'])

In [9]:
# diag_result["comp__apply_over"]
# diag_result["seg__min_length"]
# diag_result["comp__method"]
# diag_result["seg__min_length"]
# diag_result["th__k"]
diag_result["th__method"]
# diag_result["comp__temperature"]
# diag_result["comp__power"]
# diag_result["th__q"]
# diag_result["comp__method"]
# diag_result["embedding_model"]
# diag_result["sent_method"]
# diag_result["th__lambda_"]

,th__method,n_configs,n_failures,na_rate,zero_var_rate,sparse_rate,mean_total_na_pct,failure_rate
0,adaptive,28,3,0.0,0.0,0.107143,0.0,0.107143
1,top_k,56,0,0.0,0.0,0.000000,0.0,0.000000
